<a href="https://colab.research.google.com/github/yejinPARK48/michigan_building_detection/blob/main/06_Puma_estimation/Phase6_2603212.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-PUMA Building Estimation — Block-Group / PUMA-aligned

**One PUMA at a time, tiled over the PUMA polygon itself (not by county).**

Why this differs from the per-county Session A/B/C notebooks:

* PUMAs are built from 2020 census tracts, so census **block groups nest
  perfectly inside a PUMA**. Selecting the units that belong to a PUMA gives a
  boundary that matches the PUMA exactly — unlike counties, which cross PUMA
  lines.
* We therefore **tile the PUMA polygon once** with a single regular grid.
  A regular grid has no overlapping cells, so a building is never counted in
  two adjacent units (the double-counting you would get if you tiled each
  block / block group separately and they shared boundary tiles).
* Each tile is then **labelled with the block group (GEOID) it sits in** via a
  centroid spatial join, so results still aggregate to block-group and PUMA
  level downstream.

The notebook auto-selects the **smallest of the 9 PUMAs** (by land area) as the
first one to run, but you can pin a specific PUMA via `PUMA_CODE`.

NAIP imagery only — no ground-truth footprints. The seg-only U-Net predicts a
mask and buildings are counted via connected components.

**Output** (`tile_predictions_puma_<code>.csv`):
`tile_id, puma, bg_geoid, county_fips, pred_count, mask_area_m2, mask_ratio, tier`


## 0. Setup

In [ ]:
# SSL patch MUST be first, before any other network imports
import os, ssl, urllib3
os.environ['CURL_CA_BUNDLE']     = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()


In [ ]:
!pip install pystac-client planetary-computer rasterio geopandas shapely pyogrio -q
!pip install segmentation-models-pytorch albumentations opencv-python-headless scikit-image -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00


In [ ]:
import os, warnings, zipfile, urllib.request, time, shutil, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import pyproj
from pyproj import Transformer
from PIL import Image
from shapely.geometry import box, shape as shapely_shape
from shapely.ops import transform as shp_transform
from skimage import measure

import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.crs import CRS
from rasterio.warp import transform_bounds

import pystac_client
import planetary_computer as pc
pc.settings.set_subscription_key("0162e7782ab28b85cda3c77b61875ca3210a1c31")

import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from google.colab import drive
drive.mount('/content/drive')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Mounted at /content/drive
Device: cuda


## 1. Config

In [ ]:
# The 9 target PUMAs (7-digit GEOID = state '26' + 5-digit PUMA code)
PUMAS_9 = [
    '2601200', '2603203', '2600100', '2602903', '2600802',
    '2603212', '2601701', '2601703', '2601600',
]

# Leave as None to auto-select the SMALLEST PUMA (by land area) for this run.
# Set to a specific 7-digit GEOID (e.g. '2603212') to pin one.
PUMA_CODE = None

BASE_DIR      = '/content/drive/MyDrive/michigan_unet_project'
CKPT_DIR      = f'{BASE_DIR}/checkpoints'
SEG_CKPT      = f'{CKPT_DIR}/unet_seg_best_full_curated.pt'

PRED_DIR      = f'{BASE_DIR}/results_9puma/predictions_puma'
PROGRESS_DIR  = f'{BASE_DIR}/results_9puma/progress_puma'
LOCAL_IMG_DIR = '/content/tiles_tmp/images'
GEO_DIR       = f'{BASE_DIR}/geo'          # cached TIGER shapefiles

for d in [PRED_DIR, PROGRESS_DIR, LOCAL_IMG_DIR, GEO_DIR]:
    os.makedirs(d, exist_ok=True)

# Inference constants — identical to the per-county notebooks
TILE_SIZE_M    = 256
TILE_SIZE_PX   = 512
UTM_EPSG       = 26917
NAIP_YEAR      = 2020
BEST_THRESHOLD = 0.5
BEST_MIN_AREA  = 50

DENSE_THRESHOLD  = 0.05
SPARSE_THRESHOLD = 0.001
def classify_tier(ratio):
    if ratio >= DENSE_THRESHOLD:    return 'Dense'
    elif ratio >= SPARSE_THRESHOLD: return 'Sparse'
    else:                           return 'Empty'

print('Checkpoint exists:', os.path.exists(SEG_CKPT))
print('Candidate PUMAs   :', PUMAS_9)


Checkpoint exists: True
Candidate PUMAs   : ['2601200', '2603203', '2600100', '2602903', '2600802', '2603212', '2601701', '2601703', '2601600']


## 2. PUMA Geometry — load the 9 and pick the smallest

Downloads the 2020-vintage PUMA shapefile (cached to Drive). `ALAND20` is the
official land area in m^2, so we rank by it directly — no projection needed.


In [ ]:
def fetch_tiger(candidate_urls, local_zip, extract_dir):
    """Download the first reachable TIGER zip and extract it. Returns the dir."""
    if any(f.endswith('.shp') for f in os.listdir(extract_dir)):
        return extract_dir
    last_err = None
    for url in candidate_urls:
        try:
            print('Trying', url)
            urllib.request.urlretrieve(url, local_zip)
            with zipfile.ZipFile(local_zip) as z:
                z.extractall(extract_dir)
            print('  -> downloaded & extracted')
            return extract_dir
        except Exception as e:
            last_err = e
            print('  -> failed:', e)
    raise RuntimeError(f'All PUMA download URLs failed: {last_err}')

# 2020 PUMA boundaries live under .../PUMA20/. Try a couple of vintages.
PUMA_DIR = f'{GEO_DIR}/puma20'
os.makedirs(PUMA_DIR, exist_ok=True)
fetch_tiger(
    candidate_urls=[
        'https://www2.census.gov/geo/tiger/TIGER2023/PUMA20/tl_2023_26_puma20.zip',
        'https://www2.census.gov/geo/tiger/TIGER2022/PUMA20/tl_2022_26_puma20.zip',
        'https://www2.census.gov/geo/tiger/TIGER2024/PUMA20/tl_2024_26_puma20.zip',
    ],
    local_zip=f'{PUMA_DIR}/puma20.zip',
    extract_dir=PUMA_DIR,
)
puma_shp = [f'{PUMA_DIR}/{f}' for f in os.listdir(PUMA_DIR) if f.endswith('.shp')][0]
gdf_puma_all = gpd.read_file(puma_shp)

# Detect the GEOID column (7-digit state+puma) regardless of vintage suffix
geoid_col = next(c for c in gdf_puma_all.columns
                 if c.upper().startswith('GEOID'))
aland_col = next((c for c in gdf_puma_all.columns
                  if c.upper().startswith('ALAND')), None)
gdf_puma_all[geoid_col] = gdf_puma_all[geoid_col].astype(str)

gdf_puma = gdf_puma_all[gdf_puma_all[geoid_col].isin(PUMAS_9)].copy()
assert len(gdf_puma) == len(PUMAS_9), \
    f'Found {len(gdf_puma)}/{len(PUMAS_9)} PUMAs — check codes / vintage'

if aland_col:
    gdf_puma['area_km2'] = gdf_puma[aland_col] / 1e6
else:  # fall back to computing area in UTM
    gdf_puma['area_km2'] = gdf_puma.to_crs(epsg=UTM_EPSG).geometry.area / 1e6

ranking = (gdf_puma[[geoid_col, 'area_km2']]
           .sort_values('area_km2')
           .reset_index(drop=True))
ranking['est_tiles'] = (ranking['area_km2'] * 1e6 / (TILE_SIZE_M ** 2)).round().astype(int)
print('PUMAs ranked smallest -> largest (land area):')
print(ranking.to_string(index=False))

if PUMA_CODE is None:
    PUMA_CODE = ranking.iloc[0][geoid_col]
    print(f'\nAuto-selected smallest PUMA: {PUMA_CODE}')
else:
    print(f'\nUsing pinned PUMA: {PUMA_CODE}')

puma_row  = gdf_puma[gdf_puma[geoid_col] == PUMA_CODE].to_crs(epsg=4326).iloc[0]
puma_geom = puma_row.geometry
print('Selected PUMA area (km^2):',
      round(float(gdf_puma.loc[gdf_puma[geoid_col]==PUMA_CODE,'area_km2'].iloc[0]), 1))


Trying https://www2.census.gov/geo/tiger/TIGER2023/PUMA20/tl_2023_26_puma20.zip
  -> failed: HTTP Error 404: Not Found
Trying https://www2.census.gov/geo/tiger/TIGER2022/PUMA20/tl_2022_26_puma20.zip
  -> failed: HTTP Error 404: Not Found
Trying https://www2.census.gov/geo/tiger/TIGER2024/PUMA20/tl_2024_26_puma20.zip
  -> downloaded & extracted
PUMAs ranked smallest -> largest (land area):
GEOID20     area_km2  est_tiles
2603212    65.106587        993
2603203    93.779840       1431
2601703   163.192431       2490
2602903   177.935191       2715
2600802   846.509173      12917
2601701  2131.045270      32517
2601200  4417.097111      67400
2601600  6742.852277     102888
2600100 22265.787264     339749

Auto-selected smallest PUMA: 2603212
Selected PUMA area (km^2): 65.1


## 3. Block-Group Geometry — the units that nest in this PUMA

Loads the 2020 block-group shapefile (reuses the copy already in Drive if
present), then keeps only the block groups whose centroid falls inside the
selected PUMA. Their union *is* the PUMA, so every tile we keep is attributable
to exactly one block group.


In [ ]:
SHP_BG = f'{BASE_DIR}/tl_2020_26_bg.shp'
if not os.path.exists(SHP_BG):
    print('Block-group shapefile not in Drive — downloading...')
    urllib.request.urlretrieve(
        'https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_26_bg.zip',
        '/content/tl_2020_26_bg.zip')
    with zipfile.ZipFile('/content/tl_2020_26_bg.zip') as z:
        z.extractall(BASE_DIR)

gdf_bg = gpd.read_file(SHP_BG).to_crs(epsg=4326)
gdf_bg = gdf_bg.rename(columns={'GEOID': 'bg_geoid'})
gdf_bg['bg_geoid'] = gdf_bg['bg_geoid'].astype(str)

# Keep block groups whose centroid is inside this PUMA
bg_centroids = gdf_bg.copy()
bg_centroids['geometry'] = gdf_bg.geometry.centroid
in_puma = bg_centroids.geometry.within(puma_geom)
gdf_bg_puma = gdf_bg[in_puma].reset_index(drop=True)
gdf_bg_puma['county_fips'] = '26' + gdf_bg_puma['bg_geoid'].str[2:5]

print(f'Block groups in PUMA {PUMA_CODE}: {len(gdf_bg_puma)}')
print(gdf_bg_puma.groupby('county_fips').size().rename('block_groups').to_string())


Block groups in PUMA 2603212: 126
county_fips
26163    126


## 4. Tile the PUMA Polygon (single grid, no duplication)

One regular 256 m grid over the PUMA in UTM. A tile is kept if its **centroid**
lies inside the PUMA polygon — this gives exact PUMA membership with no
overlapping cells. Each kept tile is then labelled with the block group its
centroid falls in.


In [ ]:
def build_puma_tiles(puma_geom, gdf_bg_puma,
                     tile_size_m=TILE_SIZE_M, crs_utm=UTM_EPSG):
    to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{crs_utm}', always_xy=True).transform
    to_wgs = Transformer.from_crs(f'EPSG:{crs_utm}', 'EPSG:4326', always_xy=True).transform
    geom_utm = shp_transform(to_utm, puma_geom)
    minx, miny, maxx, maxy = geom_utm.bounds

    recs = []
    seq = 0
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            cell = box(x, y, x + tile_size_m, y + tile_size_m)
            c = cell.centroid
            if geom_utm.contains(c):
                recs.append({
                    'tile_id': f'{PUMA_CODE}_{seq:06d}',
                    'geometry': shp_transform(to_wgs, cell),   # WGS84 tile box
                })
                seq += 1
            y += tile_size_m
        x += tile_size_m

    gdf_t = gpd.GeoDataFrame(recs, geometry='geometry', crs='EPSG:4326')
    gdf_t['bbox'] = gdf_t.geometry.apply(lambda g: g.bounds)   # (minx,miny,maxx,maxy)

    # Label each tile with the block group its centroid sits in
    cent = gdf_t.copy()
    cent['geometry'] = gdf_t.geometry.centroid
    joined = gpd.sjoin(cent, gdf_bg_puma[['bg_geoid', 'county_fips', 'geometry']],
                       how='left', predicate='within')
    joined = joined[~joined.index.duplicated(keep='first')]
    gdf_t['bg_geoid']     = joined['bg_geoid'].values
    gdf_t['county_fips']  = joined['county_fips'].values
    # Tiles whose centroid missed a BG (rare boundary slivers): drop them
    gdf_t = gdf_t.dropna(subset=['bg_geoid']).reset_index(drop=True)
    return gdf_t

gdf_tiles = build_puma_tiles(puma_geom, gdf_bg_puma)
print(f'Tiles covering PUMA {PUMA_CODE}: {len(gdf_tiles):,}')
print(gdf_tiles.groupby('county_fips').size().rename('tiles').to_string())


Tiles covering PUMA 2603212: 1,019
county_fips
26163    1019


## 5. NAIP Fetch — search the catalog once for the whole PUMA

The per-tile STAC search is the main bottleneck. Instead we search NAIP **once**
for the PUMA bounding box, get every covering image, and assign each tile to the
image that contains its centroid. We then open each image a single time and crop
all of its tiles. Tiles not covered by any returned image fall back to a
per-tile search.


In [ ]:
def get_naip_items(bbox, year=NAIP_YEAR, max_items=2000):
    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=pc.sign_inplace)
    items = catalog.search(
        collections=['naip'], bbox=bbox,
        datetime=f'{year}-01-01/{year}-12-31', max_items=max_items
    ).item_collection()
    if len(items) == 0:
        items = catalog.search(collections=['naip'], bbox=bbox,
                               max_items=max_items).item_collection()
    return list(items)


def build_item_index(items):
    """GeoDataFrame of NAIP image footprints (unsigned href stored)."""
    recs = []
    for it in items:
        try:
            recs.append({
                'item_id': it.id,
                'href': it.assets['image'].href,   # sign right before opening
                'geometry': shapely_shape(it.geometry),
            })
        except Exception:
            continue
    return gpd.GeoDataFrame(recs, geometry='geometry', crs='EPSG:4326')


def crop_tile_from_src(src, bbox, size_px=TILE_SIZE_PX):
    """Crop one tile from an already-open NAIP dataset (no STAC search)."""
    try:
        bt = transform_bounds(CRS.from_epsg(4326), src.crs, *bbox)
        window = from_bounds(*bt, transform=src.transform)
        data = src.read([1, 2, 3], window=window,
                        out_shape=(3, size_px, size_px),
                        resampling=Resampling.bilinear)
        return np.transpose(data, (1, 2, 0)).astype(np.uint8)
    except Exception:
        return None


def fetch_naip_tile(bbox, year=NAIP_YEAR, size_px=TILE_SIZE_PX, max_retries=3):
    """Per-tile fallback: search + crop for a single tile."""
    for attempt in range(max_retries):
        try:
            catalog = pystac_client.Client.open(
                'https://planetarycomputer.microsoft.com/api/stac/v1',
                modifier=pc.sign_inplace)
            items = catalog.search(
                collections=['naip'], bbox=bbox,
                datetime=f'{year}-01-01/{year}-12-31', max_items=5
            ).item_collection()
            if len(items) == 0:
                items = catalog.search(collections=['naip'], bbox=bbox,
                                       max_items=5).item_collection()
                if len(items) == 0:
                    return None
            href = pc.sign(items[0].assets['image'].href)
            with rasterio.open(href) as src:
                return crop_tile_from_src(src, bbox, size_px)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return None
    return None

# Search once for the PUMA, then assign each tile to its covering image
puma_bbox = puma_geom.bounds
naip_items = get_naip_items(puma_bbox)
gdf_items  = build_item_index(naip_items)
print(f'NAIP {NAIP_YEAR} images covering PUMA: {len(gdf_items)}')

cent = gdf_tiles.copy()
cent['geometry'] = gdf_tiles.geometry.centroid
tile_item = gpd.sjoin(cent, gdf_items[['item_id', 'geometry']],
                      how='left', predicate='within')
tile_item = tile_item[~tile_item.index.duplicated(keep='first')]
gdf_tiles['item_id'] = tile_item['item_id'].values

n_assigned = gdf_tiles['item_id'].notna().sum()
print(f'Tiles assigned to an image: {n_assigned:,} | '
      f'fallback (per-tile search): {len(gdf_tiles) - n_assigned:,}')


NAIP 2020 images covering PUMA: 6
Tiles assigned to an image: 1,019 | fallback (per-tile search): 0


## 6. Load Trained Model

In [ ]:
model = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                 in_channels=3, classes=1, activation=None).to(DEVICE)
model.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE))
model.eval()
print('Model loaded.')

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class InferenceDataset(Dataset):
    def __init__(self, tile_ids, img_dir, transform=None):
        self.tile_ids, self.img_dir, self.transform = tile_ids, img_dir, transform
    def __len__(self): return len(self.tile_ids)
    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        img = np.array(Image.open(f'{self.img_dir}/{tid}.png').convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, tid

# 0.5 m x 0.5 m of ground per pixel (256 m tile rendered at 512 px)
PIXEL_AREA_M2 = (TILE_SIZE_M / TILE_SIZE_PX) ** 2

def count_and_area(pred_mask):
    # building count + footprint area (m^2), components >= BEST_MIN_AREA px
    labeled = measure.label(pred_mask)
    count, area_px = 0, 0
    for r in measure.regionprops(labeled):
        if r.area >= BEST_MIN_AREA:
            count += 1
            area_px += r.area
    return count, area_px * PIXEL_AREA_M2


Model loaded.


## 7. Per-Image Loop — Download -> Predict -> Save -> Delete

Processed one NAIP image at a time so disk stays flat and a Colab disconnect
resumes where it left off (each finished image is recorded in the progress
file; already-predicted tiles are skipped on restart).


In [ ]:
PRED_PATH     = f'{PRED_DIR}/tile_predictions_puma_{PUMA_CODE}.csv'
PROGRESS_PATH = f'{PROGRESS_DIR}/done_images_{PUMA_CODE}.json'

# Resume state
done_images = set()
if os.path.exists(PROGRESS_PATH):
    done_images = set(json.load(open(PROGRESS_PATH)))
done_tiles = set()
if os.path.exists(PRED_PATH):
    done_tiles = set(pd.read_csv(PRED_PATH, usecols=['tile_id'])['tile_id'])
print(f'Resuming: {len(done_images)} images / {len(done_tiles):,} tiles already done')


def predict_tiles(tile_ids):
    """Run the model over PNGs already saved in LOCAL_IMG_DIR."""
    if not tile_ids:
        return []
    ds = InferenceDataset(tile_ids, LOCAL_IMG_DIR, val_aug)
    ld = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)
    out = []
    with torch.no_grad():
        for imgs, tids in ld:
            probs  = torch.sigmoid(model(imgs.to(DEVICE)))
            masks  = (probs > BEST_THRESHOLD).cpu().numpy()
            ratios = masks.reshape(masks.shape[0], -1).mean(axis=1)
            for b in range(len(tids)):
                tid = tids[b]
                cnt, area_m2 = count_and_area(masks[b, 0])
                out.append({
                    'tile_id'      : tid,
                    'pred_count'   : cnt,
                    'mask_area_m2' : round(area_m2, 2),
                    'mask_ratio'   : round(float(ratios[b]), 6),
                    'tier'         : classify_tier(float(ratios[b])),
                })
    return out


def append_predictions(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    meta = gdf_tiles.set_index('tile_id')[['bg_geoid', 'county_fips']]
    df = df.join(meta, on='tile_id')
    df['puma'] = PUMA_CODE
    df = df[['tile_id', 'puma', 'bg_geoid', 'county_fips',
             'pred_count', 'mask_area_m2', 'mask_ratio', 'tier']]
    header = not os.path.exists(PRED_PATH)
    df.to_csv(PRED_PATH, mode='a', header=header, index=False)


# ---- assigned-image groups ----
groups = [(iid, g) for iid, g in gdf_tiles.groupby('item_id')]
total  = len(groups)
for gi, (item_id, grp) in enumerate(groups, 1):
    if item_id in done_images:
        continue
    href = gdf_items.loc[gdf_items['item_id'] == item_id, 'href'].iloc[0]
    try:
        src = rasterio.open(pc.sign(href))
    except Exception:
        src = None

    shutil.rmtree(LOCAL_IMG_DIR, ignore_errors=True)
    os.makedirs(LOCAL_IMG_DIR, exist_ok=True)

    saved, fb = [], 0
    for _, r in grp.iterrows():
        tid = r['tile_id']
        if tid in done_tiles:
            continue
        img = crop_tile_from_src(src, r['bbox']) if src is not None else None
        if img is None or img.sum() == 0:
            img = fetch_naip_tile(r['bbox'])
            if img is not None:
                fb += 1
        if img is None:
            continue
        Image.fromarray(img).save(f'{LOCAL_IMG_DIR}/{tid}.png')
        saved.append(tid)
    if src is not None:
        src.close()

    append_predictions(predict_tiles(saved))
    done_tiles.update(saved)
    done_images.add(item_id)
    json.dump(sorted(done_images), open(PROGRESS_PATH, 'w'))
    print(f'[{gi}/{total}] image {item_id} | tiles {len(saved):,} | fallback {fb}')

# ---- per-tile fallback group (tiles with no covering image) ----
no_img = gdf_tiles[gdf_tiles['item_id'].isna()]
no_img = no_img[~no_img['tile_id'].isin(done_tiles)]
if len(no_img):
    shutil.rmtree(LOCAL_IMG_DIR, ignore_errors=True)
    os.makedirs(LOCAL_IMG_DIR, exist_ok=True)
    saved = []
    for _, r in no_img.iterrows():
        img = fetch_naip_tile(r['bbox'])
        if img is None:
            continue
        Image.fromarray(img).save(f"{LOCAL_IMG_DIR}/{r['tile_id']}.png")
        saved.append(r['tile_id'])
    append_predictions(predict_tiles(saved))
    print(f'fallback group | tiles {len(saved):,}')

shutil.rmtree(LOCAL_IMG_DIR, ignore_errors=True)
os.makedirs(LOCAL_IMG_DIR, exist_ok=True)
print('\nDone. Predictions ->', PRED_PATH)


Resuming: 0 images / 0 tiles already done
[1/6] image mi_m_4208339_se_17_060_20200705 | tiles 45 | fallback 0
[2/6] image mi_m_4208340_sw_17_060_20200705 | tiles 88 | fallback 0
[3/6] image mi_m_4208347_ne_17_060_20200705 | tiles 172 | fallback 0
[4/6] image mi_m_4208347_se_17_060_20200705 | tiles 143 | fallback 0
[5/6] image mi_m_4208348_nw_17_060_20200705 | tiles 383 | fallback 0
[6/6] image mi_m_4208348_sw_17_060_20200705 | tiles 188 | fallback 0

Done. Predictions -> /content/drive/MyDrive/michigan_unet_project/results_9puma/predictions_puma/tile_predictions_puma_2603212.csv


## 8. Block-Group Aggregation + Density + Built-up Coverage

Rolls the tile-level predictions up to block group. Three kinds of measure:

* **Building count density** — predicted buildings / block-group land area
  (`building_density_km2`).
* **Built-up coverage** — total predicted building **mask area** divided by the
  block group's **total land area** (`built_fraction`, `built_pct`). Each tile
  covers 256 m x 256 m of ground, so mask area = sum over tiles of
  `mask_ratio x 0.065536 km^2`.
* **Tier mix** — how many tiles are Dense / Sparse / Empty.

`tiled_area_km2` (n_tiles x tile area) is included so you can see how completely
the tiles cover the block group; coverage relative to the *sampled* tiles only
equals `mean_mask_ratio`.

`mask_area_km2` is now summed from a per-tile `mask_area_m2` saved during
inference, using the **same >= 50 px connected components** that `pred_buildings`
counts — so footprint area and building count are consistent. (`mean_mask_ratio`
still reflects the raw thresholded mask, useful as a built-up proxy.)


In [ ]:
# ---- areas ----
TILE_AREA_KM2 = (TILE_SIZE_M ** 2) / 1e6   # 256 m x 256 m of ground per tile

# Block-group polygon area (km^2) in UTM
_bg_proj = gdf_bg_puma.to_crs(epsg=UTM_EPSG)
bg_area = pd.DataFrame({
    'bg_geoid'   : gdf_bg_puma['bg_geoid'].values,
    'county_fips': gdf_bg_puma['county_fips'].values,
    'area_km2'   : _bg_proj.geometry.area.values / 1e6,
})

df = pd.read_csv(PRED_PATH, dtype={'bg_geoid': str, 'county_fips': str})

bg_est = (df.groupby('bg_geoid')
            .agg(n_tiles=('tile_id', 'count'),
                 pred_buildings=('pred_count', 'sum'),
                 mask_area_m2=('mask_area_m2', 'sum'),
                 mean_mask_ratio=('mask_ratio', 'mean'),
                 dense_tiles=('tier', lambda s: (s == 'Dense').sum()),
                 sparse_tiles=('tier', lambda s: (s == 'Sparse').sum()),
                 empty_tiles=('tier', lambda s: (s == 'Empty').sum()))
            .reset_index())

bg_est = bg_est.merge(bg_area, on='bg_geoid', how='left')
bg_est['puma'] = PUMA_CODE

# building footprint (mask) area: summed per-tile mask area (>= 50 px, matches counts)
bg_est['mask_area_km2']  = bg_est['mask_area_m2'] / 1e6
bg_est['tiled_area_km2'] = bg_est['n_tiles'] * TILE_AREA_KM2

# built-up coverage relative to the block group's TOTAL land area
bg_est['built_fraction'] = bg_est['mask_area_km2'] / bg_est['area_km2']
bg_est['built_pct']      = bg_est['built_fraction'] * 100

# building-count density (buildings per km^2)
bg_est['building_density_km2'] = bg_est['pred_buildings'] / bg_est['area_km2']

for col, nd in [('area_km2', 4), ('tiled_area_km2', 4), ('mask_area_km2', 5),
                ('built_fraction', 5), ('built_pct', 2),
                ('building_density_km2', 2), ('mean_mask_ratio', 6)]:
    bg_est[col] = bg_est[col].round(nd)

bg_est = bg_est[['puma', 'bg_geoid', 'county_fips', 'area_km2', 'tiled_area_km2',
                 'pred_buildings', 'building_density_km2',
                 'mask_area_km2', 'built_fraction', 'built_pct',
                 'n_tiles', 'mean_mask_ratio',
                 'dense_tiles', 'sparse_tiles', 'empty_tiles']]

BG_PATH = f'{PRED_DIR}/bg_estimates_puma_{PUMA_CODE}.csv'
bg_est.to_csv(BG_PATH, index=False)
print('Block-group estimates ->', BG_PATH)
print(f'Block groups: {len(bg_est)}')
print()
print('[Top 10 block groups by built-up % (mask area / total area)]')
print(bg_est.sort_values('built_pct', ascending=False)
            .head(10)
            [['bg_geoid', 'area_km2', 'mask_area_km2', 'built_pct',
              'pred_buildings', 'building_density_km2']]
            .to_string(index=False))

# ---- PUMA rollup ----
tot_b    = bg_est['pred_buildings'].sum()
tot_a    = bg_est['area_km2'].sum()
tot_mask = bg_est['mask_area_km2'].sum()
print()
print(f'PUMA {PUMA_CODE} rollup:')
print(f'  buildings        : {tot_b:,}')
print(f'  land area        : {tot_a:.1f} km^2')
print(f'  building density : {tot_b / tot_a:.1f} per km^2')
print(f'  mask area        : {tot_mask:.2f} km^2')
print(f'  built-up         : {100 * tot_mask / tot_a:.2f} % of land')


Block-group estimates -> /content/drive/MyDrive/michigan_unet_project/results_9puma/predictions_puma/bg_estimates_puma_2603212.csv
Block groups: 126

[Top 10 block groups by built-up % (mask area / total area)]
    bg_geoid  area_km2  mask_area_km2  built_pct  pred_buildings  building_density_km2
261635262001    0.2563        0.06679      26.06             237                924.74
261635341002    0.3203        0.08258      25.78             298                930.25
261635240011    0.1917        0.04848      25.29             143                745.93
261635257004    0.2483        0.06061      24.41             184                741.06
261635314001    0.2499        0.05996      24.00             133                532.23
261635305002    0.4332        0.10386      23.98             323                745.68
261635233001    0.2209        0.04993      22.60             163                737.84
261635265001    0.5364        0.11652      21.72             375                699.07
261635

## 9. Summary

In [ ]:
df = pd.read_csv(PRED_PATH, dtype={'bg_geoid': str, 'county_fips': str})
print(f'PUMA              : {PUMA_CODE}')
print(f'Total tiles       : {len(df):,}')
print(f'Predicted buildings: {df["pred_count"].sum():,}')
print(f'Block groups      : {df["bg_geoid"].nunique()}')
print()
print('[Per block group — top 10 by buildings]')
bg = (df.groupby('bg_geoid')
        .agg(tiles=('tile_id', 'count'), buildings=('pred_count', 'sum'))
        .sort_values('buildings', ascending=False))
print(bg.head(10).to_string())
print()
print('[Tier distribution]')
print(df['tier'].value_counts().to_string())


PUMA              : 2603212
Total tiles       : 1,019
Predicted buildings: 24,107
Block groups      : 126

[Per block group — top 10 by buildings]
              tiles  buildings
bg_geoid                      
261635246001     98        642
261635348002      9        573
261635247001      9        456
261635247003      9        418
261635248002      8        399
261635348003     18        398
261635265001      9        375
261635247002     13        363
261635248001      9        356
261635231001     24        344

[Tier distribution]
tier
Dense     758
Sparse    211
Empty      50
